# Causal Inference with IV and RD

files needed = '14_files.zip' contains the following:
* data files = 'Card1995.dta', 'AL1999.dta'
* image files = 'huai1.png', 'huai2.png', 'huai3.png', 'maimonides.png'

This notebook begins our exploration of causal inference methods. We'll focus more on working with data and less on understanding every aspect of the models themselves.

\[**Note:** One package that we do not have in our econ570 environment: `linearmodels`. I use `linearmodels` here to estimate a few instrumental variables regressions, because statsmodels cannot handle them. You may need to run the following command line:
```python
!pip install linearmodels
```
Alternatively, you may simply comment out the lines of code related to `linearmodels` and `IV2SLS`.\]

In [1]:
!pip install linearmodels


In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import datetime as dt 
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
import os

os.chdir('/Users/jackson/Documents/ECON570')

# instrumental variables using two-stage least squares
from linearmodels.iv import IV2SLS

## The causal effect of education on earnings

Let's read in the Card 1995 data again.

In [3]:
# Choose the columns of interest
vars=['lwage76', 'ed76', 'age76', 'black', 'reg76r', 'smsa76r',
      'momed', 'daded', 'nomomed', 'nodaded', 'iq', 'nearc4']
wage = pd.read_stata('data/14_files/Card1995.dta',columns=vars)

# convert the int types
int_vars = ['nearc4','ed76','age76','black','reg76r','smsa76r','nomomed','nodaded']
for var in int_vars:
    wage[var]=wage[var].astype('int64')

# Potential labor force experience and squared term
wage['exp76']=wage['age76']-wage['ed76']-6
wage['exp76sq'] = wage['exp76']**2

wage.dtypes

lwage76    float32
ed76         int64
age76        int64
black        int64
reg76r       int64
smsa76r      int64
momed      float32
daded      float32
nomomed      int64
nodaded      int64
iq         float64
nearc4       int64
exp76        int64
exp76sq      int64
dtype: object

# Instrumental Variables

Let's return to our education and earnings question. First, let's follow Card (1995) by re-estimating the original Mincer OLS regression of log wages on education and experience, this time controlling for experience and several other control variables in $X_i$: indicator for Black respondents, indicator for respondents living in the South, indicator for respondents living in Standard Metropolitan Statistical Areas, and both parents' education levels (and indicators for those variables being missing). 

**Note:** estimation of this equation may not identify *causal* effects of any of the control variables on wages, so take care in interpreting those coefficient estimates as well.

$log(wage_i) = \gamma_0+\gamma_1 educ_i + \gamma_2 X_i +\epsilon_i$

In [4]:
controls = 'exp76 + exp76sq + black + reg76r + smsa76r + momed + daded + nomomed + nodaded'

In [5]:
# call statsmodels ols, specify regression, and fit with robust standard errors.
#fit method can be fit to model
# hc3 is a type of robust standard error, basically standard
res_wageed_full=smf.ols('lwage76 ~ ed76 +' + controls , data=wage).fit(cov_type='HC3')
res_wageed_full.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                lwage76   R-squared:                       0.292
Model:                            OLS   Adj. R-squared:                  0.289
Method:                 Least Squares   F-statistic:                     132.2
Date:                Thu, 06 Nov 2025   Prob (F-statistic):          2.49e-229
Time:                        11:14:43   Log-Likelihood:                -1306.1
No. Observations:                3010   AIC:                             2634.
Df Residuals:                    2999   BIC:                             2700.
Df Model:                          10                                         
Covariance Type:                  HC3                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      4.7044      0.072     64.950      0.000       4.562       4.846
ed76           0.0727      0.004     18.924      0.000       0.065       0.080
exp76          0.0839      0.007     12.332      0.000       0.071       0.097
exp76sq       -0.0023      0.000     -7.006      0.000      -0.003      -0.002
black         -0.1848      0.019     -9.801      0.000      -0.222      -0.148
reg76r        -0.1227      0.015     -7.922      0.000      -0.153      -0.092
smsa76r        0.1609      0.015     10.573      0.000       0.131       0.191
momed          0.0051      0.003      1.706      0.088      -0.001       0.011
daded         -0.0010      0.003     -0.362      0.717      -0.006       0.004
nomomed        0.0242      0.023      1.040      0.299      -0.021       0.070
nodaded       -0.0111      0.019     -0.598      0.550      -0.047       0.025
==============================================================================
Omnibus:                       60.345   Durbin-Watson:                   1.861
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               72.101
Skew:                          -0.283   Prob(JB):                     2.21e-16
Kurtosis:                       3.504   Cond. No.                     1.33e+03
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity robust (HC3)
[2] The condition number is large, 1.33e+03. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

The education coefficient estimate is 0.0727. But for reasons discussed last time, we might not trust that this represents the causal effect of education on wages.

### IV: Distance to college

Here's where Card takes a novel approach. He observes that individuals growing up near a college have significantly higher levels of education *and* earnings. Importantly, he also posits that **having a college nearby should affect earnings *only through* its effect on educational attainment**. This is the "exclusion restriction" and it is a crucial assumption in instrumental variables analysis. If this assumption is valid, local college proximity can serve as an *instrument* for education, and allow us to measure the causal effect of education on earnings.

Formally, the instrumental variables (IV) model can be written as follows:

$
\begin{align}
T_i &= \beta_1 Z_i + \beta_2 X_i + u_i\\
y_i &= \gamma_1 T_i + \gamma_2 X_i +\epsilon_i
\end{align}
$

In our context, $y_i$ is the log wage, $T_i$ is education, $X_i$ is experience and other controls, and $Z_i$ is proximity to college. The potential correlation between $\epsilon_i$ and $T_i$ can introduce bias if one simply runs an OLS regression on the second equation. The exclusion restriction is that the instrument $Z_i$ affects $y_i$ only through its effect on $T_i$. If this is assumption is satisfied, then we can "clean" $T_i$ of bias (roughly speaking) by fitting the first equation, and then use the fitted values of $T_i$ to generate an unbiased estimate in the second equation.

Practically, IV estimation can be done in two stages:
1. First, regress education on four-year college proximity `nearc4` and all other controls. Construct fitted values of education for each individual - we'll label them `ed76_fitted`.
2. Next, regress the log wage on `ed76_fitted` and all other controls.

In [6]:
# drop observations with missing log wage value
wage_sample = wage.dropna(subset=['lwage76']).copy()

In [7]:
# First stage: regress education on proximity
first_stage=smf.ols('ed76 ~ nearc4 +' + controls , data=wage_sample).fit(cov_type='HC3')
print(first_stage.summary())

# Create fitted values for education
wage_sample['ed76_fitted'] = first_stage.predict()

                            OLS Regression Results                            
Dep. Variable:                   ed76   R-squared:                       0.519
Model:                            OLS   Adj. R-squared:                  0.517
Method:                 Least Squares   F-statistic:                     393.9
Date:                Thu, 06 Nov 2025   Prob (F-statistic):               0.00
Time:                        11:14:43   Log-Likelihood:                -6134.3
No. Observations:                3010   AIC:                         1.229e+04
Df Residuals:                    2999   BIC:                         1.236e+04
Df Model:                          10                                         
Covariance Type:                  HC3                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     13.8676      0.248     56.012      0.0

In [8]:
# Second stage: regress log wages on the fitted values
second_stage=smf.ols('lwage76 ~ ed76_fitted +' + controls , data=wage_sample).fit(cov_type='HC3')
second_stage.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                lwage76   R-squared:                       0.200
Model:                            OLS   Adj. R-squared:                  0.198
Method:                 Least Squares   F-statistic:                     81.95
Date:                Thu, 06 Nov 2025   Prob (F-statistic):          2.14e-149
Time:                        11:14:43   Log-Likelihood:                -1488.6
No. Observations:                3010   AIC:                             2999.
Df Residuals:                    2999   BIC:                             3065.
Df Model:                          10                                         
Covariance Type:                  HC3                                         
===============================================================================
                  coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------
Intercept       3.7307      0.774      4.822      0.000       2.214       5.247
ed76_fitted     0.1422      0.055      2.579      0.010       0.034       0.250
exp76           0.1081      0.020      5.295      0.000       0.068       0.148
exp76sq        -0.0023      0.000     -6.750      0.000      -0.003      -0.002
black          -0.1432      0.038     -3.743      0.000      -0.218      -0.068
reg76r         -0.1072      0.020     -5.250      0.000      -0.147      -0.067
smsa76r         0.1342      0.026      5.117      0.000       0.083       0.186
momed          -0.0041      0.008     -0.509      0.611      -0.020       0.012
daded          -0.0078      0.006     -1.282      0.200      -0.020       0.004
nomomed         0.0495      0.031      1.593      0.111      -0.011       0.110
nodaded        -0.0062      0.021     -0.301      0.764      -0.046       0.034
==============================================================================
Omnibus:                       46.297   Durbin-Watson:                   1.837
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               55.262
Skew:                          -0.236   Prob(JB):                     1.00e-12
Kurtosis:                       3.466   Cond. No.                     1.44e+04
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity robust (HC3)
[2] The condition number is large, 1.44e+04. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

**From the second stage, the point estimate for $\beta$ is now 0.1422 !** This result is substantially larger than our OLS estimate. That is, the cross-sectional earnings gap between more and less educated workers *understates* the causal effect of effect of education on earnings.

### Standard errors

Are we done? Not yet - the standard errors resulting from our second regression are wrong, as they don't account for the imprecision of our first stage predictions. To get the correct standard errors, let's use IV2SLS from `linearmodels`, which we imported above.

Notice the special formula syntax below. `nearc4` is an instrument for `ed76`. I've also altered the `cov_type` argument to suit the `linearmodels` syntax.

In [9]:
# construct string formula
iv_formula = 'lwage76 ~ 1 + [ed76 ~ nearc4] +' + controls
# estimate iv regression
iv_card=IV2SLS.from_formula(iv_formula, wage_sample).fit(cov_type = 'robust')
print(iv_card)

                          IV-2SLS Estimation Summary                          
Dep. Variable:                lwage76   R-squared:                      0.2066
Estimator:                    IV-2SLS   Adj. R-squared:                 0.2039
No. Observations:                3010   F-statistic:                    845.20
Date:                Thu, Nov 06 2025   P-value (F-stat)                0.0000
Time:                        11:14:43   Distribution:                 chi2(10)
Cov. Estimator:                robust                                         
                                                                              
                             Parameter Estimates                              
            Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
------------------------------------------------------------------------------
Intercept      3.7307     0.7904     4.7200     0.0000      2.1816      5.2799
exp76          0.1081     0.0209     5.1759     0.00

This gives the same point estimates running two OLS regressions, but now we get accurate standard errors.

# Practice

Let's now consider the issue of omitted variable bias in a new context: the effects of class size on academic achievement. In particular, we'll follow [Angrist and Lavy (1999)](http://piketty.pse.ens.fr/files/AngristLavy1999.pdf) to estimate the effect of elementary school class size on reading scores in Israeli public schools.

1. Use `pd.read_stata()` to bring in data set 'AL1999.dta' and name the data frame `reading`. Select just a few columns of interest:
```python
columns=['enrollment', 'grade', 'classize', 'avgverb', 'disadvantaged']
```
Keep only observations from 5th grade classrooms.

2. Run an OLS regression of `avgverb ~ classize` and display your results. How do you interpret the results? What is the main coefficient of interest? Discuss with your neighbor!

3. Next, run an OLS regression of `avgverb ~ classize + disadvantaged + enrollment` and display your results. What happened to the coefficient of interest? Why do you think this happened? Discuss with your neighbor!

In [10]:
reading = pd.read_stata("data/14_files/AL1999.dta")
columns = ['enrollment', 'grade', 'classize', 'avgverb', 'disadvantaged']
reading
reading = reading.loc[reading['grade'] == 5]
reading.shape

(2018, 31)

In [12]:
reading

,schlcode,enrollment,enrollment_boys,enrollment_girls,c_num4rd,c_type,flgrm4,mrkgrm4,ngrm4,flmth4,...,avgverb,passverb,disadvantaged,c_num5rd,flgrm5,mrkgrm5,ngrm5,flmth5,mrkmth5,nmth5
2049,11005,54,24,30,NaN,2,NaN,NaN,NaN,NaN,...,70.570000,35.700001,24,2.0,27.270000,73.0,55.0,20.000000,73.0,55.0
2050,11005,54,24,30,NaN,2,NaN,NaN,NaN,NaN,...,75.000000,18.500000,24,2.0,27.270000,73.0,55.0,20.000000,73.0,55.0
2051,11006,37,21,16,NaN,2,NaN,NaN,NaN,NaN,...,75.470001,20.000000,38,2.0,31.430000,67.0,35.0,48.570000,56.0,35.0
2052,11006,37,21,16,NaN,2,NaN,NaN,NaN,NaN,...,60.647499,40.002499,38,2.0,31.430000,67.0,35.0,48.570000,56.0,35.0
2053,11009,32,17,15,NaN,1,NaN,NaN,NaN,NaN,...,73.970001,40.599998,6,1.0,40.630001,74.0,32.0,21.879999,68.0,32.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4062,61351,19,8,11,NaN,1,NaN,NaN,NaN,NaN,...,81.290001,5.900000,26,1.0,5.880000,81.0,17.0,29.410000,69.0,17.0
4063,61363,27,14,13,NaN,1,NaN,NaN,NaN,NaN,...,79.360001,18.200001,22,1.0,18.180000,79.0,22.0,13.640000,70.0,22.0
4064,61364,70,42,28,NaN,2,NaN,NaN,NaN,NaN,...,77.360001,27.299999,0,2.0,29.030001,78.0,62.0,24.590000,69.0,61.0
4065,61364,70,42,28,NaN,2,NaN,NaN,NaN,NaN,...,81.529999,31.600000,0,2.0,29.030001,78.0,62.0,24.590000,69.0,61.0


In [11]:
res_read = smf.ols('avgverb ~ classize', data=reading).fit(cov_type='HC3')
res_read.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                avgverb   R-squared:                       0.036
Model:                            OLS   Adj. R-squared:                  0.035
Method:                 Least Squares   F-statistic:                     60.52
Date:                Thu, 06 Nov 2025   Prob (F-statistic):           1.15e-14
Time:                        11:14:43   Log-Likelihood:                -6940.9
No. Observations:                2018   AIC:                         1.389e+04
Df Residuals:                    2016   BIC:                         1.390e+04
Df Model:                           1                                         
Covariance Type:                  HC3                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     67.7197      0.917     73.811      0.000      65.922      69.518
classize       0.2225      0.029      7.780      0.000       0.166       0.279
==============================================================================
Omnibus:                      155.182   Durbin-Watson:                   1.278
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              210.143
Skew:                          -0.648   Prob(JB):                     2.33e-46
Kurtosis:                       3.906   Cond. No.                         144.
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity robust (HC3)
"""

# Regression Discontinuity

Another method commonly used to measure a causal effect is called regression discontinuity. Let's consider the following illustrative example. See [here](https://climate.uchicago.edu/wp-content/uploads/2024/10/New-evidence-on-the-impact-of-sustained-exposure-to-air-pollution_Research-Highlights.pdf) for further details.

What is the causal effect of air quality on life expectancy?

In the 1950s, China introduced a mandate for all those living north of the Huai River to use coal for indoor heating. The Chinese government also heavily subsidized the cost of coal for these individuals. In sharp contrast, those living south of the Huai River faced no such mandate and received no subsidies. This Huai River Policy created a large discontinuity in coal use around the Huai River boundary. The following choropleth map shows the PM10 particulate matter levels across China, with the Huai River running east/west.

<img src="huai1.png" alt= “” width=500 height=500>

The idea is to measure differences in an outcome of interest right around the discontinuity to measure the difference. Below, the authors estimate the discontinuity in particulate matter north and south of the river.

<img src="huai2.png" alt= “” width=500 height=500>

We can do the same thing with life expectancy just north and south of the river.

<img src="huai3.png" alt= “” width=500 height=500>

Under the right assumptions, we can use these discontinuities to estimate the effect of exposure to increased particulate matter levels on life expectancy.

## "Fuzzy" Regression Discontinuity

We can use a similar approach to estimate the *causal* effect of class size on reading test scores, using a unique schooling policy in Israel. According to the [Maimonides' rule](https://en.wikipedia.org/wiki/Maimonides%27_rule), class sizes are capped at 40 students.

In order to see how this rule generates a discontinuity, consider the max class size relative to total enrollment. With total enrollment of 120, there may be three classes with exactly 40 students in each. With enrollment of 121, three classes can no longer satisfy Maimonides' rule. So there must be four classes, each with significantly fewer than 40 students each. There's a discontinuity in class sizes with respect to total enrollment. Do we observe this discontinuity in practice?

<img src="maimonides.png" alt= “” width=500 height=500>

We can see that the rule affects class sizes, but the discontinuity in class size is not "sharp." This is what we call a "fuzzy" regression discontinuity. Instead of calculating the causal effect based on a sharp change in the outcome at the discontinuity, we can use instrumental variables tools to generate a causal estimate. We'll use the discontinuity as an *instrument* for class size in our regression model. 

## The "running" variable

Regression discontinuity strategies have various strategies for controlling for the effects of the "running" variable; that is, the variable that generates the discontinuity. In the pollution example, the running variable is distance north of the Huai River. North-south geographic location may have other effects on health, other than through this specific energy policy. In the Maimonides example, the running variable is enrollment. School enrollment may impact student outcomes by, e.g., impacting school investments or attracting different student populations. 

Common approaches:

1. Control for the running variable, or a flexible function of the running variable (e.g., a polynomial), directly.
2. *Zoom in* on a narrow bandwidth around the discontinuity. A full treatment of how to choose the right bandwidth is outside the scope of this class. If you want to try this, reach out to me!
3. Check that nothing else of importance changes at the discontinuity. Identification requires that all other factors, observed *and* unobserved, trend smoothly at the cutoff, but we can at least check observed factors.

# Practice

1. Return to the Angrist and Lavy (1999) data. Create a new variable that calculates the predicted class size based on Maimonides' rule:
```python
reading['fsc'] = reading['enrollment'] / ((reading['enrollment'] - 1) // 40 + 1)
```

2. Let's consider the "fuzzy RD" approach discussed above. This is an IV estimator, so let's first use the two-stage approach from earlier:
- First, regress `classize` on `fsc` and control for `disadvantaged` and `enrollment`. Does the instrument have the power to predict `classize`?
- Take fitted values of `classize` for each classroom from the regression.
- Next, regress the `avgverb` on the fitted values `classize_fitted` and control for `disadvantaged` and `enrollment`.

In [ ]:
reading['fsc'] = reading['enrollment'] / ((reading['enrollment']- 1) // 40 + 1)
res_read_fsc = smf.ols('avgverb ~ fsc', data=reading).fit(cov_type='HC3')   
res_read_fsc.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                avgverb   R-squared:                       0.010
Model:                            OLS   Adj. R-squared:                  0.009
Method:                 Least Squares   F-statistic:                     15.89
Date:                Thu, 06 Nov 2025   Prob (F-statistic):           6.95e-05
Time:                        11:20:25   Log-Likelihood:                -6968.1
No. Observations:                2018   AIC:                         1.394e+04
Df Residuals:                    2016   BIC:                         1.395e+04
Df Model:                           1                                         
Covariance Type:                  HC3                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     70.5707      1.003     70.389      0.000      68.606      72.536
fsc            0.1231      0.031      3.986      0.000       0.063       0.184
==============================================================================
Omnibus:                      193.215   Durbin-Watson:                   1.269
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              277.959
Skew:                          -0.738   Prob(JB):                     4.39e-61
Kurtosis:                       4.063   Cond. No.                         163.
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity robust (HC3)
"""

3. Since we want accurate standard errors, reestimate using `IV2SLS`.

4. How do we interpret the results? Discuss with your neighbor!

### Bonus Practice - Try at Home

5. Try plotting the Maimonides rule "formula" over a scatterplot of class size vs. enrollment data. Can we replicate the figure from Angrist and Lavy?

6. Create a scatterplot of `disadvantaged` vs. enrollment. Are there any sharp changes in the share of disadvantaged students at the discontinuities for us to worry about?

7. The paper includes richer controls for enrollment, and presents specifications that "Zoom in" on schools within 5 students of the Maimonides' rule cutoffs: 40, 80, 120, ... Try subsetting your data to only include schools close to the cutoffs. What happens to your estimates?